# Brain-expressed CNV convergence

Use `CNV_NETWORK=PCNET2.2_brain` (default) or `STRING_high_conf_brain`.
HPA expression and region files are frozen in `input/expression/HPA_brain/`. Retain genes with **TPM > 1 in any of the 13 brain regions**, using the TPM column (not nTPM or pTPM). Exact unambiguous HPA gene symbols map to network Entrez identifiers. Unmapped or ambiguous genes are excluded.
The induced graph is restricted to its largest connected component **before propagation**; seeds and null references use that universe. Sets with fewer than 3 retained seeds are excluded. Inspect `data/network_configs/<network>/brain_filter.json` and run seed QC for losses.
Caches are under `data/cache/PCNET2.2_brain/` or `data/cache/STRING_high_conf_brain/`; results are under the corresponding `results/<network>/`. Original workflows retain their own paths. Completed results are reused by content identity.
Downloads are identified by checksums; the unversioned HPA URL is not assumed to be the historical v24 release.


In [ ]:
import os
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from cnv_overlap.pipeline import AnalysisConfig, run
from cnv_overlap.visualization import pair_heatmap, size_scatter


ROOT = Path.cwd()
if ROOT.name == "notebooks":  # opened from notebooks/ in Jupyter
    ROOT = ROOT.parent
if not (ROOT / "configs").is_dir():
    raise RuntimeError("Start this notebook with the project root as working directory")
CONFIG_PATH = Path(os.environ.get("CNV_CONFIG", ROOT / "configs/cnv_convergence.json"))
import sys
if str(ROOT / "scripts") not in sys.path:
    sys.path.insert(0, str(ROOT / "scripts"))
from network_inputs import load_network_inputs
# Select PCNET2.2_brain or STRING_high_conf_brain.
NETWORK = os.environ.get("CNV_NETWORK", "PCNET2.2_brain")
if not NETWORK.endswith("_brain"):
    NETWORK += "_brain"
# Minimum seeds a gene set needs inside the selected network; smaller sets are excluded.
MIN_SEEDS = 3
spec, root, inputs, provenance = load_network_inputs(CONFIG_PATH, NETWORK, min_seeds=MIN_SEEDS)
RESULTS_ROOT = (root / spec["output_root"]).parent
print("Network:", NETWORK or "custom snapshot", "Results:", RESULTS_ROOT, "Cache:", root / spec["cache_dir"])
config = AnalysisConfig.from_dict(spec["analysis"])
if config.comparisons != "cnv_cnv":
    raise ValueError("This notebook requires analysis.comparisons=cnv_cnv")
A, nodes, symbols, entities, coordinates, regions = inputs
print(f"{len(nodes):,} nodes, {A.nnz // 2:,} edges; {len(entities)} gene sets")
print("Supporting pairs: overlap size; NetColoc normal-tail p-values; overlap policy:", config.overlap)
print("Gene-label permutations:", config.permutation_reps)

from notebook_analysis import SCZ_TARGETS, run_convergence


In [ ]:
RUN = run(*inputs, config=config, output_root=root/spec["output_root"],
          cache_dir=root/spec["cache_dir"], provenance=provenance)
manifest = json.loads((RUN / "manifest.json").read_text())
assert manifest["status"] == "complete"
print(RUN)
print((RUN / "summary.md").read_text())
FIGURES = RUN.parent.parent / "figures" / RUN.name
FIGURES.mkdir(parents=True, exist_ok=True)


## Input QC and tested hypotheses

Inspect seed losses, coordinate coverage and the hypothesis list. Coordinates document CNV content; all retained network genes are eligible for degree-matched references.
Only CNV loci contribute to the tested family.


In [ ]:
qc = pd.read_csv(RUN / "seed_qc.tsv", sep="\t")
display(qc)
display(pd.read_csv(RUN / "seed_overlap.tsv", sep="\t").set_index("entity"))
display(pd.read_csv(RUN / "hypotheses.tsv", sep="\t").groupby("family").size().rename("tests"))


## A. Multi-CNV convergence: the primary biological question

A gene is reached when NPS exceeds the configured individual threshold (default 1). Count distinct CNVs reaching each gene, without filtering CNV pairs by significance. The global statistic sums `k*(k-1)/2` across genes, where `k` is the number of reaching CNVs: it measures repeated convergence at the same genes.

For each joint randomization, independently sample a seed set for every CNV, preserving its exact seed count in each network-degree bin, and propagate all sets. Compare the observed statistic and gene counts with these null draws. The kernel and degree-profile references are reused; checkpoint blocks resume interrupted simulations.

`p_empirical` at each gene is unadjusted. `p_maxT` compares its count with the largest count anywhere in each null draw. This is simultaneous calibration under the complete joint seed null, **not a claim of strong gene-level FWER control under arbitrary partial alternatives**. Count statistics are discrete; ties and conservative tails are expected. All genes are exported.

The global SCZ endpoints sum the same convergence weights over genes reached by each **fixed** SCZ profile. No convergence-gene significance filter is used to select these tests. Their two empirical p-values are BH-adjusted together. These test localization of CNV convergence to a fixed SCZ network footprint; they do not randomize SCZ genetics or establish clinical specificity.

In [ ]:
# Load SCZ targets using the validated snapshot, with the same tested network.
scz_entities = {k: v for k, v in entities.items() if k in SCZ_TARGETS and v["kind"] == "target"}
if "CNV_CONFIG" not in os.environ:
    from cnv_overlap.provenance import graph_hash
    scz_spec, scz_root, scz_inputs, scz_provenance = load_network_inputs(ROOT / "configs/cnv_disorder_overlap.json", NETWORK, min_seeds=MIN_SEEDS)
    if graph_hash(scz_inputs[0], scz_inputs[1]) != graph_hash(A, nodes):
        raise ValueError("SCZ targets require the same network universe as CNV convergence")
    scz_entities = {k: v for k, v in scz_inputs[3].items() if k in SCZ_TARGETS and v["kind"] == "target"}
if not scz_entities:
    print("No schema/gwaspri targets in these custom inputs: SCZ relevance endpoints are unavailable.")
JOINT_REPS = config.permutation_reps
CONVERGENCE = run_convergence(inputs, config=config, cache_dir=root/spec["cache_dir"],
    output_root=(root/spec["output_root"]).parent/"multi_cnv_convergence/degree", reps=JOINT_REPS, scz_entities=scz_entities)
convergence_genes = pd.read_csv(CONVERGENCE / "gene_convergence.tsv", sep="\t")
convergence_tests = pd.read_csv(CONVERGENCE / "convergence_tests.tsv", sep="\t")
display(convergence_tests)
display(convergence_genes.sort_values(["n_cnvs", "p_empirical"], ascending=[False, True]).head(50))
print("Full results:", CONVERGENCE)
fig, ax = plt.subplots(figsize=(7, 4))
convergence_genes.n_cnvs.value_counts().sort_index().plot.bar(ax=ax)
ax.set(xlabel="Number of CNVs reaching a gene", ylabel="Network genes", title="Multi-CNV recurrence: all network genes")
fig.tight_layout()
fig.savefig(FIGURES / "multi_cnv_recurrence.png", dpi=200)
plt.show()


## CNV × SCHEMA × GWAS: where the networks meet

For each CNV, a gene is kept only if it lies in **both** the CNV × SCHEMA and the CNV × GWAS colocalised sets (NPS above the individual and product thresholds for the CNV and for each target). `n_cnvs_both` counts the CNVs that meet this at the same gene; the primary shared core keeps genes with `n_cnvs_both >= 3`. Genes reached by SCHEMA alone or GWAS alone do not count. `n_cnvs_schema` and `n_cnvs_gwaspri` give the single-target counts.

In [ ]:
from shared_network_pipeline import run_shared_network_pipeline
from shared_network_enrichment import LIBRARIES
MIN_SHARED_CNVS = 3
network_entities = entities | scz_entities
SCZ_ENRICHMENT = None
SCZ_SENSITIVITY = None
if any(n in network_entities and network_entities[n]["kind"] == "target" for n in SCZ_TARGETS):
    annotation_dir = root/"input/annotations"
    missing_libraries = [n for n in LIBRARIES.values() if not (annotation_dir/n).exists()]
    both_targets = all(n in network_entities and network_entities[n]["kind"] == "target" for n in SCZ_TARGETS)
    do_enrichment = both_targets and not missing_libraries
    if missing_libraries:
        print("Shared-core enrichment unavailable; missing local libraries:", missing_libraries)
    workflow = run_shared_network_pipeline(
        (A, nodes, symbols, network_entities, coordinates, regions), config=config,
        cache_dir=root/spec["cache_dir"], output_root=RESULTS_ROOT,
        annotation_dir=annotation_dir, min_cnvs=MIN_SHARED_CNVS, enrich=do_enrichment, collapse=False, )
    SCZ_NETWORKS, SCZ_ENRICHMENT = workflow["networks"], workflow["enrichment"]
    SCZ_SENSITIVITY = workflow["sensitivity"]
    network_index = pd.read_csv(SCZ_NETWORKS/"network_index.tsv", sep="\t")
    display(network_index)
    shared_genes = pd.read_csv(SCZ_NETWORKS/"gene_membership.tsv", sep="\t")
    if "shared_core" in shared_genes:
        display(shared_genes.loc[shared_genes.shared_core].sort_values("n_cnvs_both", ascending=False))
        from IPython.display import Image
        core = network_index[network_index.kind == "shared_core"].iloc[0]
        display(Image(filename=str(SCZ_NETWORKS/core.directory/"network.png")))
    else:
        print("Joint core unavailable: both SCHEMA and GWAS-prioritized targets are required.")
    print("Complete network exports:", SCZ_NETWORKS)
else:
    print("SCZ networks unavailable: no SCHEMA/GWAS-prioritized target sets supplied.")


### Primary core (≥3 CNVs): size test, function network and gene × CNV matrix
- **Size vs permutations**: the observed core size against the 5,000 joint degree-matched CNV seed draws from the sensitivity run (SCZ profiles fixed). This tests only the number of core genes; genes themselves have no significance.
- **Function network**: colour = broad function category, size = number of CNVs with SCHEMA + GWAS at the gene. A gene takes the first category, in fixed priority order, whose GO/SynGO term-name pattern matches a term annotating it; `gene_functions.tsv` lists the term used. Categories are descriptive labels, not enrichment results. The 9-colour CNV pie network above is kept as the companion plot.
- **Gene × CNV matrix**: dark tile = the CNV colocalises with both SCHEMA and GWAS at the gene; S / G = only one target; open circle = the gene lies inside that CNV.

In [ ]:
if globals().get("SCZ_NETWORKS") is not None and SCZ_SENSITIVITY is not None:
    from IPython.display import Image
    from shared_core_figures import shared_core_figures
    SCZ_FIGURES = shared_core_figures(globals().get("SCZ_NETWORKS"), sensitivity_dir=SCZ_SENSITIVITY,
        annotation_dir=annotation_dir, output_root=RESULTS_ROOT/"scz_core_figures")
    display(pd.read_csv(SCZ_FIGURES/"core_size_test.tsv", sep="\t"))
    display(Image(filename=str(SCZ_FIGURES/"core_size_null.png")))
    display(Image(filename=str(SCZ_FIGURES/"function_network.png")))
    display(Image(filename=str(SCZ_FIGURES/"gene_cnv_matrix.png")))
    print("CNV-coloured pie network:", open(SCZ_FIGURES/"pies_network.txt").read().strip())
    print("Function assignments (with the GO/SynGO term used), tile states and figures:", SCZ_FIGURES)
else:
    print("Core figures require both SCHEMA and GWAS targets and the core sensitivity run.")


### Leiden clusters and biological annotations
Partition the final shared CNV × SCHEMA × GWAS core using Leiden (seed 42, resolution 1). Annotate each cluster with GO and Reactome using the eligible network background and BH correction across all cluster–term tests. These are descriptive annotations after network selection. Singletons (genes with no edge in the core, and any size-1 community) are excluded from clustering and cluster enrichment and are listed in `excluded_singletons.tsv`.


In [ ]:
SCZ_LEIDEN = None
if "SCZ_NETWORKS" in globals() and both_targets:
    from shared_network_leiden import annotate_leiden
    SCZ_LEIDEN = annotate_leiden(SCZ_NETWORKS, annotation_dir=annotation_dir,
        output_root=RESULTS_ROOT/"scz_network_leiden", seed=42, resolution=1.0, fdr=config.fdr)
    display(pd.read_csv(SCZ_LEIDEN/"clusters.tsv", sep="\t"))
    print("Excluded singletons:", ", ".join(pd.read_csv(SCZ_LEIDEN/"excluded_singletons.tsv", sep="\t").symbol.astype(str)))
    from IPython.display import Image
    display(Image(filename=str(SCZ_LEIDEN/"leiden_clusters.png")))
    print("Annotated Leiden graph, memberships and enrichment:", SCZ_LEIDEN)
else:
    print("Leiden shared-core analysis requires both SCHEMA and GWAS targets.")


### Network context: pathways, synapse, tissue and cell type
Annotate every SCZ colocalised network (each CNV × SCHEMA and CNV × GWAS pair, each CNV with both targets, and the shared core) using the frozen GO (BP, CC, MF), Reactome, SynGO 2024, Allen human MTG cell-type markers and Human Protein Atlas gene sets. The HPA sets hold the genes that HPA calls elevated (enriched, group enriched or enhanced) in each tissue, brain region, brain single-nucleus cell type and body-wide single-cell type. The test is hypergeometric ORA against all network genes, with BH correction within each annotation group (function, synapse, tissue, brain region, cell type) across all colocalised networks. `gene_context.tsv` gives the HPA specificity of every gene in any colocalised network and lists the networks that contain it. These annotations are descriptive and are not adjusted for network selection. The libraries are frozen by `scripts/fetch_cluster_annotations.py`.

In [ ]:
SCZ_CONTEXT = None
if "SCZ_NETWORKS" in globals() and both_targets:
    from shared_network_cluster_context import annotate_network_context
    SCZ_CONTEXT = annotate_network_context(SCZ_NETWORKS, annotation_dir=annotation_dir,
        output_root=RESULTS_ROOT/"scz_network_cluster_context", fdr=config.fdr)
    with pd.option_context("display.max_colwidth", 200, "display.max_rows", None):
        display(pd.read_csv(SCZ_CONTEXT/"network_context.tsv", sep="\t")[
            ["network", "n_genes", "top_function", "top_synapse", "top_tissue", "top_brain_region", "top_cell_type"]])
    display(pd.read_csv(SCZ_CONTEXT/"gene_context.tsv", sep="\t"))
    from IPython.display import Image
    display(Image(filename=str(SCZ_CONTEXT/"tissue_celltype_enrichment.png")))
    print("Colocalised-network context tables and tissue/cell-type heatmap:", SCZ_CONTEXT)
else:
    print("Network context requires the SCZ colocalised networks with both SCHEMA and GWAS targets.")


## Shared-core threshold sensitivity and matching null

The primary core remains ≥3 CNVs with **both** SCHEMA and GWAS at the same gene for each contributing CNV. Sensitivity uses ≥2, ≥3 and ≥4 CNVs with SCHEMA, GWAS, either, or both targets. Each joint degree-matched CNV seed draw rebuilds every definition, keeping SCZ profiles fixed. Empirical P-values test core size; BH Q-values cover all 12 variants. This is an exploratory extension, without gene-level significance. Enrichment across variants is descriptive and corrected across all variant–library terms.


In [ ]:
if SCZ_SENSITIVITY is not None:
    from IPython.display import Image
    core_tests = pd.read_csv(SCZ_SENSITIVITY/"core_size_tests.tsv", sep="\t")
    display(core_tests.drop(columns="genes"))
    display(Image(filename=str(SCZ_SENSITIVITY/"core_sensitivity.png")))
    if (SCZ_SENSITIVITY/"enrichment_significant.tsv").exists():
        sensitivity_terms = pd.read_csv(SCZ_SENSITIVITY/"enrichment_significant.tsv", sep="\t")
        display(sensitivity_terms.sort_values("q_all_variants_libraries").head(40))
    print("Sensitivity memberships, null draws and enrichment:", SCZ_SENSITIVITY)
else:
    print("Core sensitivity requires both SCHEMA and GWAS targets.")


### Shared-network pathways, biological processes and molecular functions

In [ ]:
if SCZ_ENRICHMENT is not None:
    from IPython.display import Image
    enrichment = pd.read_csv(SCZ_ENRICHMENT/"enrichment_all_terms.tsv", sep="\t")
    display(pd.read_csv(SCZ_ENRICHMENT/"annotation_coverage.tsv", sep="\t"))
    for library in LIBRARIES:
        print(library)
        display(enrichment[(enrichment.library == library) & (enrichment.q_all_libraries < config.fdr)]
                [["term", "overlap", "query_size", "fold_enrichment", "q_all_libraries", "genes"]])
    display(Image(filename=str(SCZ_ENRICHMENT/"enrichment_summary.png")))
    print("All enrichment results:", SCZ_ENRICHMENT)
else:
    print("Shared-network enrichment was not run; see target/library availability above.")


## Relaxed shared core: ≥2 CNVs with SCHEMA and GWAS

The same definition as the primary core, with the threshold lowered to **≥2 CNVs**: each contributing CNV must colocalise with **both** SCHEMA and GWAS at the gene. The core-size test against the joint degree-matched null for this variant is the `both`, `min_cnvs = 2` row of the sensitivity table above. The network plot, enrichment, Leiden clusters and context annotation below are descriptive and not adjusted for network selection. Results are written to separate `min2cnvs` directories, so the primary ≥3 outputs are unchanged.

In [ ]:
SCZ_NETWORKS_2 = SCZ_ENRICHMENT_2 = None
if "SCZ_NETWORKS" in globals() and both_targets:
    from IPython.display import Image
    workflow_2 = run_shared_network_pipeline(
        (A, nodes, symbols, network_entities, coordinates, regions), config=config,
        cache_dir=root/spec["cache_dir"], output_root=RESULTS_ROOT,
        annotation_dir=annotation_dir, min_cnvs=2, enrich=do_enrichment, collapse=False,
        sensitivity=False)  # the ≥2 variant is already in the sensitivity null above
    SCZ_NETWORKS_2, SCZ_ENRICHMENT_2 = workflow_2["networks"], workflow_2["enrichment"]
    index_2 = pd.read_csv(SCZ_NETWORKS_2/"network_index.tsv", sep="\t")
    core_2 = index_2[index_2.kind == "shared_core"].iloc[0]
    display(core_2.to_frame().T)
    genes_2 = pd.read_csv(SCZ_NETWORKS_2/"gene_membership.tsv", sep="\t")
    genes_2 = genes_2.loc[genes_2.shared_core].assign(in_primary_core=lambda d: d.n_cnvs_both >= MIN_SHARED_CNVS)
    print(f"{len(genes_2)} genes at >=2 CNVs; {genes_2.in_primary_core.sum()} of them are also in the >={MIN_SHARED_CNVS} core.")
    with pd.option_context("display.max_rows", None):
        display(genes_2.sort_values(["n_cnvs_both", "symbol"], ascending=[False, True])[
            ["symbol", "n_cnvs_both", "cnvs_both", "n_cnvs_schema", "n_cnvs_gwaspri", "gene_origin", "in_primary_core"]])
    display(Image(filename=str(SCZ_NETWORKS_2/core_2.directory/"network.png")))
    print("Complete >=2 network exports:", SCZ_NETWORKS_2)
else:
    print("The relaxed core requires both SCHEMA and GWAS targets.")


### ≥2-CNV core: size test, function network and gene × CNV matrix
- **Size vs permutations**: the observed core size against the 5,000 joint degree-matched CNV seed draws from the sensitivity run (SCZ profiles fixed). This tests only the number of core genes; genes themselves have no significance.
- **Function network**: colour = broad function category, size = number of CNVs with SCHEMA + GWAS at the gene. A gene takes the first category, in fixed priority order, whose GO/SynGO term-name pattern matches a term annotating it; `gene_functions.tsv` lists the term used. Categories are descriptive labels, not enrichment results. The 9-colour CNV pie network above is kept as the companion plot.
- **Gene × CNV matrix**: dark tile = the CNV colocalises with both SCHEMA and GWAS at the gene; S / G = only one target; open circle = the gene lies inside that CNV.

In [ ]:
if SCZ_NETWORKS_2 is not None and SCZ_SENSITIVITY is not None:
    from IPython.display import Image
    from shared_core_figures import shared_core_figures
    SCZ_FIGURES_2 = shared_core_figures(SCZ_NETWORKS_2, sensitivity_dir=SCZ_SENSITIVITY,
        annotation_dir=annotation_dir, output_root=RESULTS_ROOT/"scz_core_figures")
    display(pd.read_csv(SCZ_FIGURES_2/"core_size_test.tsv", sep="\t"))
    display(Image(filename=str(SCZ_FIGURES_2/"core_size_null.png")))
    display(Image(filename=str(SCZ_FIGURES_2/"function_network.png")))
    display(Image(filename=str(SCZ_FIGURES_2/"gene_cnv_matrix.png")))
    print("CNV-coloured pie network:", open(SCZ_FIGURES_2/"pies_network.txt").read().strip())
    print("Function assignments (with the GO/SynGO term used), tile states and figures:", SCZ_FIGURES_2)
else:
    print("Core figures require both SCHEMA and GWAS targets and the core sensitivity run.")


### ≥2-CNV core: pathways, biological processes and molecular functions

In [ ]:
if SCZ_ENRICHMENT_2 is not None:
    from IPython.display import Image
    enrichment_2 = pd.read_csv(SCZ_ENRICHMENT_2/"enrichment_all_terms.tsv", sep="\t")
    display(pd.read_csv(SCZ_ENRICHMENT_2/"annotation_coverage.tsv", sep="\t"))
    for library in LIBRARIES:
        print(library)
        display(enrichment_2[(enrichment_2.library == library) & (enrichment_2.q_all_libraries < config.fdr)]
                [["term", "overlap", "query_size", "fold_enrichment", "q_all_libraries", "genes"]])
    display(Image(filename=str(SCZ_ENRICHMENT_2/"enrichment_summary.png")))
    print("All >=2 enrichment results:", SCZ_ENRICHMENT_2)
else:
    print("Enrichment of the >=2 core was not run; see target/library availability above.")


### ≥2-CNV core: Leiden clusters and network context
The same Leiden settings (seed 42, resolution 1) and context libraries as for the primary core. The context table is restricted to the shared core; the pair and per-CNV networks are identical to the primary run.

In [ ]:
if SCZ_NETWORKS_2 is not None:
    from IPython.display import Image
    from shared_network_leiden import annotate_leiden
    from shared_network_cluster_context import CORE, annotate_network_context
    SCZ_LEIDEN_2 = annotate_leiden(SCZ_NETWORKS_2, annotation_dir=annotation_dir,
        output_root=RESULTS_ROOT/"scz_network_leiden", seed=42, resolution=1.0, fdr=config.fdr)
    display(pd.read_csv(SCZ_LEIDEN_2/"clusters.tsv", sep="\t"))
    print("Excluded singletons:", ", ".join(pd.read_csv(SCZ_LEIDEN_2/"excluded_singletons.tsv", sep="\t").symbol.astype(str)))
    display(Image(filename=str(SCZ_LEIDEN_2/"leiden_clusters.png")))
    print("Annotated >=2 Leiden graph:", SCZ_LEIDEN_2)
    SCZ_CONTEXT_2 = annotate_network_context(SCZ_NETWORKS_2, annotation_dir=annotation_dir,
        output_root=RESULTS_ROOT/"scz_network_cluster_context", fdr=config.fdr)
    context_2 = pd.read_csv(SCZ_CONTEXT_2/"network_context.tsv", sep="\t")
    with pd.option_context("display.max_colwidth", 200):
        display(context_2[context_2.network == CORE][
            ["network", "n_genes", "top_function", "top_synapse", "top_tissue", "top_brain_region", "top_cell_type"]])
    display(Image(filename=str(SCZ_CONTEXT_2/"tissue_celltype_enrichment.png")))
    print(">=2 context tables and heatmap:", SCZ_CONTEXT_2)
else:
    print("Leiden and context analyses of the >=2 core require both SCHEMA and GWAS targets.")


## Leave-one-CNV-out stability

Recompute observed convergence after removing each CNV. This is a descriptive influence check, not nine additional significance tests or independent replication. Inspect whether the global signal and SCZ-related convergence are dominated by one CNV. The gene export lists all contributing CNVs, allowing individual genes to be checked too.

In [ ]:
leave_one_out = pd.read_csv(CONVERGENCE / "leave_one_out.tsv", sep="\t")
display(leave_one_out)
# A descriptive community network built from genes reached by at least two CNVs.
from cnv_overlap.interpretation import candidate_network
multi_candidates = convergence_genes.loc[convergence_genes.n_cnvs >= 2, ["entrez"]]
MULTI_G, multi_membership = candidate_network(A, nodes, multi_candidates)
multi_membership = multi_membership.merge(convergence_genes, on="entrez", how="left")
multi_membership.to_csv(FIGURES / "multi_cnv_communities.tsv", sep="\t", index=False)
display(multi_membership.head(40))


## Optional contiguous-gene-window sensitivity (off by default)

In [ ]:
RUN_REGION_SENSITIVITY = False
if RUN_REGION_SENSITIVITY:
    REGION_REPS = min(1000, config.permutation_reps)
    REGION_CONVERGENCE = run_convergence(inputs, config=config, cache_dir=root/spec["cache_dir"],
        output_root=(root/spec["output_root"]).parent/"multi_cnv_convergence/region", reps=REGION_REPS,
        null_model="region", scz_entities=scz_entities)
    region_tests = pd.read_csv(REGION_CONVERGENCE / "convergence_tests.tsv", sep="\t")
    display(pd.concat([convergence_tests.assign(null_model="degree"),
                       region_tests.assign(null_model="contiguous_gene_window")], ignore_index=True))
    display(pd.read_csv(REGION_CONVERGENCE / "seed_qc.tsv", sep="\t"))
else:
    print("Optional contiguous-gene-window sensitivity not run.")


## Supporting CNV–CNV pair tests

Stars mark BH-adjusted significance for the size endpoint within this notebook’s comparison family. Effects are the NetColoc COLOC score, observed / expected network size (expected = mean over gene-label permutations; colour on a log2 scale around 1), with significance from the normal (Z) test against the permutation null and BH correction. This tests network colocalization; it does not establish genomic-region or disease specificity.


`p` and `q` use the NetColoc normal approximation. `p_empirical` and the tail intervals diagnose the finite permutation distribution; they are not confidence intervals for the normal-tail p-value. Constant nulls have missing p/q.


In [ ]:
tests = pd.read_csv(RUN / "pair_tests.tsv", sep="\t")
PRIMARY_NULL = "netcoloc"
primary = tests[(tests["null"] == PRIMARY_NULL) & (tests.statistic == "size")].copy()
for family in ["region_pair"]:
    if (primary.family == family).any():
        fig = pair_heatmap(RUN, null=PRIMARY_NULL, family=family, fdr=config.fdr)
        fig.savefig(FIGURES / f"{family}.png", bbox_inches="tight")
        fig.savefig(FIGURES / f"{family}.pdf", bbox_inches="tight")
        plt.show()
        fig = pair_heatmap(RUN, null=PRIMARY_NULL, family=family, fdr=config.fdr, metric="z")
        fig.savefig(FIGURES / f"{family}_z.png", bbox_inches="tight")
        fig.savefig(FIGURES / f"{family}_z.pdf", bbox_inches="tight")
        plt.show()
        fig = size_scatter(RUN, null=PRIMARY_NULL, family=family, fdr=config.fdr)
        fig.savefig(FIGURES / f"{family}_sizes.png", bbox_inches="tight")
        fig.savefig(FIGURES / f"{family}_sizes.pdf", bbox_inches="tight")
        plt.show()
display(primary.assign(size_ratio=primary.observed / primary.null_mean).sort_values(["family", "q"])[["a", "b", "family", "observed", "null_mean", "size_ratio", "excess", "p", "q", "p_empirical", "tail_ci_low", "tail_ci_high", "status"]])


## Brain-focused Enrichr analysis of significant CNV × CNV pairs

Each query is the shared, thresholded candidate-gene set of a **significant CNV × CNV pair** (NetColoc size BH q < config.fdr); non-significant CNV pairs are not tested. CNV × SCHEMA and CNV × GWAS are shown in the disorder-overlap notebook.

Use the **Enrichr custom-background API (Speedrichr)** with GO Biological Process 2026 and SynGO 2024. Reactome uses the **official June 2026 GMT** (`input/annotations/ReactomePathways.gmt`, the same release as the rest of the project); Enrichr only offers Reactome 2024, so the identical one-sided hypergeometric test is computed locally with the same background (it matches Enrichr's p-values to within 1%). SynGO provides brain-focused synaptic annotation; GO and Reactome are tested in the same brain-network background. This is not a claim that every GO/Reactome term is brain-specific.

The background is the actual propagated brain-network universe; shared seeds are removed only for `overlap=remove`. Terms need ≥5 background genes. Enrichr p-values are BH-corrected **across all tested pairs and all three libraries**, including zero-hit terms. Enrichr's own per-library adjusted p-values are also retained. Enrichment is descriptive after pair/network selection, not selection-adjusted evidence.

One plot shows every significant CNV × CNV pair (rows). Columns, per library, are every row's 2 strongest significant terms, filled to 12 with the strongest terms overall (ordered by best combined q), so a strong pair cannot hide the others; columns are coloured by library; dot size is −log10(combined BH q). Blank positions have no significant enrichment.

**Term collapsing (orsum 1.8.0).** For each plot and library, orsum receives every row's significant terms as one ranked list and picks representative terms jointly across rows, using the full annotation sets. Each column is a representative; **(+n)** counts the further terms it represents, and a dot shows the best original q among them. q-values are never recomputed. Reactome collapses strongly because its gene sets are nested; Enrichr's GO BP sets are not propagated up the GO hierarchy, so GO collapses only modestly. Uncollapsed plots are saved as `*_all_terms.png`, and `*_term_mapping.tsv` lists every representative with the terms it covers. Raw API responses, downloaded GMTs, full result tables, coverage, query genes and backgrounds are cached alongside PNG/PDF figures. The first run submits query/background gene symbols to Enrichr; subsequent runs reuse cached responses.

In [ ]:
from brain_pair_enrichment import enrich_pairs
from IPython.display import Image
BRAIN_PAIR_ENRICHMENT = enrich_pairs(
    RUN, annotation_dir=root / "input/annotations",
    output_root=RESULTS_ROOT / "brain_pair_enrichment_enrichr" / config.comparisons,
    fdr=config.fdr, backend="enrichr",
)
print((BRAIN_PAIR_ENRICHMENT / "summary.md").read_text())
display(pd.read_csv(BRAIN_PAIR_ENRICHMENT / "tested_pairs.tsv", sep="\t")[["a", "b", "pair_group", "q", "significant"]])
for figure in ["enrichment_cnv_cnv"]:
    display(Image(filename=str(BRAIN_PAIR_ENRICHMENT / f"{figure}.png")))
display(pd.read_csv(BRAIN_PAIR_ENRICHMENT / "enrichment_significant.tsv", sep="\t").head(40))
print("Complete brain enrichment results:", BRAIN_PAIR_ENRICHMENT)


## Secondary profile summaries and gene candidates

Signed similarity can include shared avoidance. These statistics do not select primary discoveries. Candidate genes meet fixed joint-score thresholds; they have no inferred genome-wide gene-level q-values.

In [ ]:
display(tests[tests.statistic != "size"].sort_values(["null", "statistic", "q"]).head(30))
candidates = pd.read_csv(RUN / "candidate_genes.tsv", sep="\t")
selected = set(zip(primary.loc[primary.q < config.fdr, "a"], primary.loc[primary.q < config.fdr, "b"]))
selected_candidates = candidates.loc[np.array([pair in selected for pair in zip(candidates.a, candidates.b)], dtype=bool)]
display(selected_candidates.head(40))
recurrence = pd.read_csv(RUN / "recurrence.tsv", sep="\t")
display(recurrence.sort_values(["n_regions", "n_pairs"], ascending=False).head(30))


## Descriptive communities

Each connected component is partitioned separately. Distinct components are never pooled into a fictitious community zero. Recurrence, degree and community membership describe the candidate network; they are not independent replication or significance tests.

In [ ]:
from cnv_overlap.interpretation import candidate_network
G, membership = candidate_network(A, nodes, selected_candidates)
membership["symbol"] = membership.entrez.map(symbols)
membership.to_csv(FIGURES / "candidate_communities.tsv", sep="\t", index=False)
print(f"{G.number_of_nodes()} candidate nodes, {G.number_of_edges()} edges")
display(membership.sort_values(["community", "degree_in_candidate_network"], ascending=[True, False]).head(40))


## Network of significant CNV × CNV pairs

Pool the thresholded candidate genes of every **significant CNV × CNV pair** (NetColoc size BH q < config.fdr) into one union network, induced on the tested brain network. Each gene records the significant pairs and CNVs that support it (`gene_membership.tsv`); genes supported by several pairs are where independent pairwise convergences meet.

Partition the union with Leiden (per connected component, seed 42, resolution 1); isolated genes and size-1 communities are excluded and listed in `excluded_singletons.tsv`. The whole network and every cluster with ≥ 5 genes are annotated by hypergeometric ORA against all network genes, using the same frozen libraries as the SCZ context analysis (GO BP/CC/MF, Reactome, SynGO 2024, HPA tissue, brain region and cell type, Allen MTG), BH-corrected within each annotation group across all tested gene sets. `cluster_pair_composition.tsv` gives, for each cluster, the share of its genes that are candidates of each significant pair, showing which CNV pairs drive which module.

Descriptive only: genes and clusters carry no significance of their own, and enrichment is not adjusted for pair or network selection. The significant pairs differ between networks, so compare modules, not cluster numbers.


In [ ]:
from cnv_pair_network import build_pair_network
from IPython.display import Image
PAIR_NETWORK = build_pair_network(RUN, inputs, annotation_dir=root/"input/annotations",
    output_root=RESULTS_ROOT/"cnv_pair_network", fdr=config.fdr, null=PRIMARY_NULL)
pair_network_genes = pd.read_csv(PAIR_NETWORK/"gene_membership.tsv", sep="\t")
pair_clusters = pd.read_csv(PAIR_NETWORK/"clusters.tsv", sep="\t")
display(pd.read_csv(PAIR_NETWORK/"significant_pairs.tsv", sep="\t"))
display(Image(filename=str(PAIR_NETWORK/"cnv_pair_graph.png")))
print(f"{len(pair_network_genes)} union genes, {(pair_network_genes.n_pairs >= 2).sum()} supported by ≥ 2 significant pairs; "
      f"{len(pd.read_csv(PAIR_NETWORK/'edges.tsv', sep='\t'))} edges; {len(pair_clusters)} Leiden clusters")
display(pair_network_genes[pair_network_genes.n_pairs >= 2].drop(columns="degree_in_background").head(40))
with pd.option_context("display.max_colwidth", 120):
    display(pair_clusters.drop(columns="genes"))
display(Image(filename=str(PAIR_NETWORK/"leiden_clusters.png")))
display(Image(filename=str(PAIR_NETWORK/"cluster_pair_composition.png")))
with pd.option_context("display.max_colwidth", 200, "display.max_rows", None):
    display(pd.read_csv(PAIR_NETWORK/"cluster_context.tsv", sep="\t")[
        ["cluster", "n_genes", "top_function", "top_synapse", "top_tissue", "top_brain_region", "top_cell_type"]])
display(Image(filename=str(PAIR_NETWORK/"tissue_celltype_enrichment.png")))
print("Union network, memberships, clusters, enrichment and figures:", PAIR_NETWORK)


## Offline annotation and rewiring robustness

Run annotation with explicit GMT snapshots using `cnv-overlap annotate`; query/background use the same selected network universe. Enrichment is descriptive ORA after network selection. Brain-filtered inputs and the retained network universe are used throughout this notebook.

In [ ]:
from notebook_analysis import export_versioned_report
GMT_PATHS = [root / "input/annotations" / name for name in
             ["GO_Biological_Process_2026.gmt", "ReactomePathways.gmt"]]
if not all(path.exists() for path in GMT_PATHS):
    print("Annotation snapshots missing; exporting figures and networks only.")
    GMT_PATHS = []
REPORT = export_versioned_report(RUN, inputs, output_dir=RUN.parent.parent / "reports" / RUN.name,
                       gmt_paths=GMT_PATHS, fdr=config.fdr)
print("Heatmaps, pair networks and annotation outputs:", REPORT)
if GMT_PATHS:
    annotation = pd.read_csv(REPORT / "enrichment/descriptive_ora.tsv", sep="\t")
    selected_pairs = primary.loc[primary.q < config.fdr, ["a", "b"]]
    displayed = annotation.merge(selected_pairs, on=["a", "b"])
    display(displayed[displayed.q_all_pair_terms < config.fdr].sort_values("q_all_pair_terms").head(40))
